In [31]:
import time
import torch
import torch.nn.functional as F
from PIL import Image
import importlib
import schedulers.scheduling_ddpm
import schedulers.scheduling_ddim
import schedulers
import pipelines.ddpm
import pipelines

importlib.reload(schedulers.scheduling_ddpm)   # parent first
importlib.reload(schedulers.scheduling_ddim)   # then the subclass
importlib.reload(schedulers)                   # re-export the new classes
importlib.reload(pipelines.ddpm)
importlib.reload(pipelines)

from models import UNet
from schedulers import DDPMScheduler, DDIMScheduler
from pipelines import DDPMPipeline

# Quick sanity checks created by Claude

In [32]:
torch.manual_seed(0)
T = 1000

# ---------------------------------------------------------------- 0. quick sanity checks
sched = DDPMScheduler(num_train_timesteps=T, beta_start=1e-4, beta_end=0.02)
ac = sched.alphas_cumprod

# the schedule goes from "almost all image" to "almost all noise"
assert ac[0] > 0.999 and ac[-1] < 1e-4 and (ac[1:] < ac[:-1]).all(), "schedule is wrong"

# add_noise must equal sqrt(abar)*x0 + sqrt(1-abar)*noise
x0 = torch.randn(4, 3, 8, 8)
noise = torch.randn_like(x0)
t = torch.tensor([0, 10, 500, 999])
a = ac[t].view(-1, 1, 1, 1)
assert torch.allclose(sched.add_noise(x0, noise, t), a.sqrt() * x0 + (1 - a).sqrt() * noise, atol=1e-5), \
    "add_noise does not match the formula (check the sqrt(1 - abar) coefficient)"

# noising keeps unit variance when x0 has unit variance
big = torch.randn(64, 3, 8, 8)
std = sched.add_noise(big, torch.randn_like(big), torch.full((64,), 500)).std().item()
assert abs(std - 1) < 0.05, f"variance not preserved at t=500 (std={std:.2f})"

# DDIM with eta=0 has no randomness: same input -> same output
ddim_check = DDIMScheduler(num_train_timesteps=T, num_inference_steps=200, beta_start=1e-4, beta_end=0.02)
eps = torch.randn_like(x0)
assert torch.equal(ddim_check.step(eps, torch.tensor(500), x0), ddim_check.step(eps, torch.tensor(500), x0))

# Demo by Claude. How to use the code implemented in Part A.

In [33]:

# ---------------------------------------------------------------- 1. forward process
print("Part 1: forward process. 'image' = sqrt(abar), 'noise' = sqrt(1-abar)")
for step in (0, 100, 250, 500, 750, 999):
    ab = ac[step].item()
    print(f"  t={step:4d}   image weight {ab ** 0.5:.3f}   noise weight {(1 - ab) ** 0.5:.3f}")
print()


# ---------------------------------------------------------------- 2. tiny training loop
def make_batch(n):
    """Toy images: a horizontal or vertical ramp, randomly flipped in sign. Values in [-1, 1]."""
    ramp = torch.linspace(-1, 1, 8)
    horiz = ramp.view(1, 1, 1, 8).expand(n, 3, 8, 8)
    vert = ramp.view(1, 1, 8, 1).expand(n, 3, 8, 8)
    pick = torch.rand(n, 1, 1, 1) < 0.5
    sign = (torch.rand(n, 1, 1, 1) < 0.5).float() * 2 - 1
    return torch.where(pick, horiz, vert) * sign


unet = UNet(input_size=8, input_ch=3, T=T, ch=32, ch_mult=[1, 2], attn=[1], num_res_blocks=1)
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-3)

print("Part 2: training (each step = noise an image, predict the noise, MSE)")
unet.train()
for step in range(600):
    images = make_batch(64)                              # clean images x0
    noise = torch.randn_like(images)                     # the noise WE choose (the answer)
    timesteps = torch.randint(0, T, (64,)).long()        # a random timestep per image
    noisy = sched.add_noise(images, noise, timesteps)    # forward process
    pred = unet(noisy, timesteps)                        # UNet guesses the noise
    loss = F.mse_loss(pred, noise)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
    optimizer.step()
    if step % 100 == 0 or step == 599:
        print(f"  step {step:3d}   loss {loss.item():.4f}")
print()

# ---------------------------------------------------------------- 3. sampling
unet.eval()
rows = {"data": [Image.fromarray(((img.permute(1, 2, 0) / 2 + 0.5).clamp(0, 1) * 255).byte().numpy())
                 for img in make_batch(8)]}

# DDPM: random noise at every step; 200 steps instead of 1000 (the pipeline calls set_timesteps for us)
pipe = DDPMPipeline(unet, sched)
start = time.time()
rows["DDPM 200 steps"] = pipe(batch_size=8, num_inference_steps=200, generator=torch.Generator().manual_seed(0),
                              device="cpu")
print(f"Part 3: DDPM, 200 steps: {time.time() - start:.1f}s")

# DDIM: deterministic, only 50 steps. It needs num_inference_steps when it is created.
ddim = DDIMScheduler(num_train_timesteps=T, num_inference_steps=50, beta_start=1e-4, beta_end=0.02)
pipe = DDPMPipeline(unet, ddim)
start = time.time()
rows["DDIM 50 steps"] = pipe(batch_size=8, num_inference_steps=50, generator=torch.Generator().manual_seed(0),
                             device="cpu")
print(f"        DDIM, 50 steps:  {time.time() - start:.1f}s  (same UNet, ~4x fewer evaluations)")

# save a picture: one row per method, images enlarged 16x
scale = 16
grid = Image.new("RGB", (8 * 8 * scale, len(rows) * 8 * scale))
for r, (name, imgs) in enumerate(rows.items()):
    for c, img in enumerate(imgs):
        grid.paste(img.resize((8 * scale, 8 * scale), Image.NEAREST), (c * 8 * scale, r * 8 * scale))
grid.save("demo_samples.png")
print("\nSaved demo_samples.png (rows: data, DDPM, DDIM). Generated rows should resemble ramps, loosely.")

Part 1: forward process. 'image' = sqrt(abar), 'noise' = sqrt(1-abar)
  t=   0   image weight 1.000   noise weight 0.010
  t= 100   image weight 0.946   noise weight 0.324
  t= 250   image weight 0.722   noise weight 0.692
  t= 500   image weight 0.279   noise weight 0.960
  t= 750   image weight 0.057   noise weight 0.998
  t= 999   image weight 0.006   noise weight 1.000

Part 2: training (each step = noise an image, predict the noise, MSE)
  step   0   loss 0.9986
  step 100   loss 0.0554
  step 200   loss 0.0361
  step 300   loss 0.0395
  step 400   loss 0.0334
  step 500   loss 0.0280
  step 599   loss 0.0268



100%|██████████| 200/200 [00:05<00:00, 35.25it/s]


Part 3: DDPM, 200 steps: 5.7s


100%|██████████| 50/50 [00:01<00:00, 40.15it/s]

        DDIM, 50 steps:  1.2s  (same UNet, ~4x fewer evaluations)

Saved demo_samples.png (rows: data, DDPM, DDIM). Generated rows should resemble ramps, loosely.
